In [2]:
# FAERS Drug Severity Analysis
# Project Overview

# This notebook analyzes adverse event reports from the FDA Adverse Event Reporting System (FAERS) to explore potential relationships between reported drugs and the severity of patient outcomes.

# The analysis uses a report–drug pair dataset constructed from the FAERS relational database. Each row represents a drug that appears within a specific adverse event report, along with contextual information about that report.

# The dataset was generated using a SQL pipeline that joins several FAERS tables:

# drug — drugs reported in each case

# reac — adverse reactions associated with the report

# outc — outcome classifications (death, hospitalization, etc.)

# demo — report-level metadata

# The SQL script that generates the dataset can be found in:

# sql/build_faers_severity_dataset_table.sql

# The resulting table (faers_severity_dataset) contains report-level context features and outcome indicators for downstream analysis.

# Dataset Structure

# Each row represents one drug within one FAERS report.

# Key columns include:

# Column	Description
# primaryid	Unique identifier for the FAERS report
# drugname	Drug name reported in the case
# role_cod	Role of the drug in the report (PS = primary suspect, SS = secondary suspect, C = concomitant)
# n_drugs	Number of drugs reported in the case
# n_reactions	Number of reactions reported in the case
# death_flag	Whether the report includes death as an outcome
# hosp_flag	Whether hospitalization was reported
# life_threat_flag	Whether the event was life-threatening
# serious_flag	Indicator of whether the report contains a serious outcome
# Analysis Goals

# The goals of this analysis are to:

# Explore the distribution of serious outcomes in FAERS reports.

# Identify drugs associated with higher proportions of serious adverse events.

# Investigate how report complexity (number of drugs or reactions) relates to outcome severity.

# Prepare the dataset for downstream statistical testing and machine learning modeling.

# Project Workflow

# The full analysis pipeline follows this structure:

# SQL feature engineering

# Build a clean report–drug analysis dataset from the FAERS database.

# Exploratory data analysis

# Inspect dataset structure and distributions.

# Identify high-frequency drugs and reaction patterns.

# Statistical analysis

# Evaluate associations between drugs and serious outcomes.

# Machine learning

# Build predictive models for serious adverse event reports.

# Important Notes

# FAERS is a voluntary reporting system, and therefore:

# Reports may contain missing or incomplete information.

# Report frequency does not imply causal relationships.

# The results of this analysis should be interpreted as signal exploration, not definitive safety conclusions.



#FAERS Raw Tables
#    │
#    ├── drug   (reported medications)
#    ├── reac   (reported adverse reactions)
#    ├── outc   (outcome classifications)
#    └── demo   (report metadata)
#    │
#    ▼
# SQL Feature Engineering
# (sql/build_faers_severity_dataset_table.sql)
#    │
#    ▼
# Create report–drug dataset with:
# - case complexity features
# - outcome severity flags
# - drug role information
#    │
#    ▼
# SQLite Database Table
# faers_severity_dataset
#    │
#    ▼
# Python Analysis Notebook
# (notebooks/faers_severity_analysis.ipynb)

# Steps performed in this notebook:
# - Dataset inspection
# - Exploratory analysis
# - Statistical testing
# - Machine learning modeling


from pathlib import Path
import sqlite3
import pandas as pd

In [3]:

project_root = Path().resolve().parent

db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)

print("Connected to:", db_path)

Connected to: C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db


In [ ]:
query = """
SELECT *
FROM faers_severity_dataset
"""

df = pd.read_sql_query(query, conn)



# -------------------------------------------------------------
# Handle missing outcome flags
# -------------------------------------------------------------
#
# Some FAERS reports do not contain entries in the `outc` table.
# Because the SQL pipeline used LEFT JOINs, these cases appear
# with NULL values for outcome indicators.
#
# In this context, missing values indicate that no coded
# serious outcome was recorded for the report. Therefore,
# we replace NULL values with 0 for all outcome flags.
# -------------------------------------------------------------

df["serious_flag"] = df["serious_flag"].fillna(0)
df["death_flag"] = df["death_flag"].fillna(0)
df["hosp_flag"] = df["hosp_flag"].fillna(0)
df["life_threat_flag"] = df["life_threat_flag"].fillna(0)

df.head()

,primaryid,drugname,role_cod,n_drugs,n_reactions,death_flag,hosp_flag,life_threat_flag,serious_flag
0,1001678125,SANDOSTATIN,PS,42,154.0,0.0,0.0,0.0,0.0
1,1001678125,SANDOSTATIN,SS,42,154.0,0.0,0.0,0.0,0.0
2,1001678125,SANDOSTATIN,SS,42,154.0,0.0,0.0,0.0,0.0
3,1001678125,SANDOSTATIN LAR DEPOT,SS,42,154.0,0.0,0.0,0.0,0.0
4,1001678125,SANDOSTATIN LAR DEPOT,SS,42,154.0,0.0,0.0,0.0,0.0


In [11]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 9645822 entries, 0 to 9645821
Data columns (total 9 columns):
 #   Column            Dtype  
---  ------            -----  
 0   primaryid         int64  
 1   drugname          str    
 2   role_cod          str    
 3   n_drugs           int64  
 4   n_reactions       float64
 5   death_flag        float64
 6   hosp_flag         float64
 7   life_threat_flag  float64
 8   serious_flag      float64
dtypes: float64(5), int64(2), str(2)
memory usage: 662.3 MB


In [12]:
df.shape

(9645822, 9)

In [13]:
df["serious_flag"].mean()

np.float64(0.46466397576069723)

In [16]:
# -------------------------------------------------------------
# Calculate serious outcome rate by drug
# -------------------------------------------------------------
#
# This step aggregates the dataset at the drug level in order
# to estimate how frequently reports involving each drug are
# associated with serious outcomes.
#
# The FAERS dataset currently contains one row per:
#     (report, drug) pair
#
# Therefore we group by `drugname` to summarize across all
# reports that mention each drug.
#
# Metrics calculated:
#
# reports
#     Total number of FAERS report–drug pairs involving
#     this drug in the dataset.
#
# serious_rate
#     Proportion of those reports that contain a serious
#     outcome. Since `serious_flag` is binary (0 or 1),
#     the mean directly represents the serious outcome rate.
#
# This aggregation allows us to identify drugs that appear
# disproportionately associated with severe adverse events.
# Later we will filter out low-frequency drugs to avoid
# misleading rates from very small sample sizes.
# -------------------------------------------------------------

drug_severity = (
    df.groupby("drugname")
      .agg(
          reports=("drugname", "count"),
          serious_rate=("serious_flag", "mean")
      )
      .reset_index()
)

drug_severity.head()

,drugname,reports,serious_rate
0,Oxybutymin,1,1.0
1,LATANOPROST EYE DROPS,2,1.0
2,1000iu vitamin D,2,0.0
3,5-HTP,1,0.0
4,ACTOVASTATIN,2,0.0


In [17]:
df["serious_flag"].isna().sum()

np.int64(0)

In [22]:
# normalize names first
df["drugname"] = df["drugname"].str.upper().str.strip()

# then aggregate
drug_severity = (
    df.groupby("drugname")
      .agg(
          reports=("drugname", "count"),
          serious_rate=("serious_flag", "mean")
      )
      .reset_index()
)

In [25]:
drug_severity_filtered = drug_severity[
    drug_severity["reports"] >= 500
]

drug_severity_filtered.shape


(1939, 3)

In [27]:
# -------------------------------------------------------------
# Filter drugs with sufficient report counts
# -------------------------------------------------------------
#
# Drugs appearing only a few times in FAERS can produce
# unstable severity estimates (e.g., 1 report = 100% serious).
#
# To reduce statistical noise, we restrict the analysis to
# drugs with at least 500 reported cases.
#
# This threshold ensures that calculated serious outcome
# rates are based on a meaningful sample size.
# -------------------------------------------------------------



top_serious_drugs = drug_severity_filtered.sort_values(
    "serious_rate",
    ascending=False
)

top_serious_drugs.head(20)

,drugname,reports,serious_rate
50040,"SODIUM PHOSPHATE, MONOBASIC",636,1.000000
2354,ALISKIREN HEMIFUMARATE,607,0.985173
3330,AMITRIPTYLINE HYDROCHLORIDE\PERPHENAZINE,601,0.983361
18193,DISTIGMINE BROMIDE,691,0.979740
49975,SODIUM FERRIC GLUCONATE COMPLEX,1977,0.978756
1753,AGGRENOX,836,0.971292
30687,LANTHANUM CARBONATE,1283,0.971161
43022,PERSANTINE,548,0.967153
29021,ISOPROPYL ALCOHOL,2642,0.966692
487,ACEBUTOLOL HYDROCHLORIDE,1900,0.957895


In [28]:
top_serious_drugs.shape

(1939, 3)

In [30]:
# -------------------------------------------------------------
# Build contingency table counts for each drug
# -------------------------------------------------------------
#
# a = serious reports involving the drug
# b = non-serious reports involving the drug
#
# These values will later be used to compute the
# Reporting Odds Ratio (ROR).
# -------------------------------------------------------------

drug_counts = (
    df.groupby("drugname")
      .agg(
          a_serious=("serious_flag", "sum"),
          total_reports=("serious_flag", "count")
      )
)

drug_counts["b_non_serious"] = (
    drug_counts["total_reports"] - drug_counts["a_serious"]
)

drug_counts

,a_serious,total_reports,b_non_serious
drugname,,,
"(6S)-5-METHYLTETRAHYDROFOLATE, ASCORBIC ACID, BETACAROTENE, BIOTIN, BO",0.0,1,1.0
(6S)-5-METHYLTETRAHYDROFOLATE;COLECALCIFEROL;CYANOCOBALAMIN;FOLIC ACID,1.0,1,0.0
(RS)-3 METHYL-2-OXOVALERIANIC ACID CALCIUM,0.0,1,1.0
(SARS-COV-2) VACCINE,0.0,2,2.0
(UNSWEETENED) HIBISCUS TEA AS SIALOGOGUE,0.0,2,2.0
...,...,...,...
^KONAKION MM,2.0,2,0.0
^LORB^ ORAL BIRTH CONTROL,0.0,1,1.0
^METROGEL,0.0,2,2.0


In [37]:
# -------------------------------------------------------------
# Reporting Odds Ratio (ROR) Signal Detection
# -------------------------------------------------------------
#
# The Reporting Odds Ratio (ROR) is a pharmacovigilance metric
# used to detect potential safety signals in spontaneous
# reporting systems such as FAERS.
#
# It compares the odds of a serious outcome for a specific drug
# against the odds of serious outcomes for all other drugs in
# the dataset.
#
# The contingency table structure:
#
#                    Serious       Non-Serious
# Drug of interest       a              b
# Other drugs            c              d
#
# ROR = (a * d) / (b * c)
#
# Where:
#
# a = serious reports involving the drug
# b = non-serious reports involving the drug
# c = serious reports involving all other drugs
# d = non-serious reports involving all other drugs
#
# Interpretation:
#
# ROR > 1  → drug appears more frequently in serious reports
# ROR = 1  → no difference from background reporting rate
# ROR < 1  → drug appears less frequently in serious reports
#
# IMPORTANT:
# Very small cell counts can cause division-by-zero errors
# or unstable ratios. To stabilize the calculation we apply
# the Haldane–Anscombe correction, adding 0.5 to each cell
# of the contingency table. This is standard practice in
# pharmacovigilance analyses.
#
# Note:
# ROR identifies statistical signals but does NOT establish
# causation.
# -------------------------------------------------------------


# Compute global totals across the dataset
total_serious = df["serious_flag"].sum()
total_reports = len(df)
total_non_serious = total_reports - total_serious


# Compute counts for "other drugs"
drug_counts["c_other_serious"] = (
    total_serious - drug_counts["a_serious"]
)

drug_counts["d_other_non_serious"] = (
    total_non_serious - drug_counts["b_non_serious"]
)


# Compute Reporting Odds Ratio with continuity correction
drug_counts["ROR"] = (
    (drug_counts["a_serious"] + 0.5) *
    (drug_counts["d_other_non_serious"] + 0.5)
) / (
    (drug_counts["b_non_serious"] + 0.5) *
    (drug_counts["c_other_serious"] + 0.5)
)


# Convert index to column
drug_ror = drug_counts.reset_index()


# Filter to drugs with sufficient report counts
drug_ror = drug_ror[drug_ror["total_reports"] >= 500]


# Inspect strongest signals
drug_ror.sort_values("ROR", ascending=False)[
    ["drugname", "total_reports", "a_serious", "b_non_serious", "ROR"]
].head(20)



drug_ror = drug_counts.reset_index()

drug_ror = drug_ror[drug_ror["total_reports"] >= 500]

drug_ror.sort_values("ROR", ascending=False).head(20)

,drugname,a_serious,total_reports,b_non_serious,c_other_serious,d_other_non_serious,ROR
50040,"SODIUM PHOSPHATE, MONOBASIC",636.0,636,0.0,4481430.0,5163756.0,1466.822262
2354,ALISKIREN HEMIFUMARATE,598.0,607,9.0,4481468.0,5163747.0,72.591404
3330,AMITRIPTYLINE HYDROCHLORIDE\PERPHENAZINE,591.0,601,10.0,4481475.0,5163746.0,64.909660
18193,DISTIGMINE BROMIDE,677.0,691,14.0,4481389.0,5163742.0,53.838529
49975,SODIUM FERRIC GLUCONATE COMPLEX,1935.0,1977,42.0,4480131.0,5163714.0,52.489895
30687,LANTHANUM CARBONATE,1246.0,1283,37.0,4480820.0,5163719.0,38.305939
1753,AGGRENOX,812.0,836,24.0,4481254.0,5163732.0,38.213904
29021,ISOPROPYL ALCOHOL,2554.0,2642,88.0,4479512.0,5163668.0,33.272868
43022,PERSANTINE,530.0,548,18.0,4481536.0,5163738.0,33.040831
487,ACEBUTOLOL HYDROCHLORIDE,1820.0,1900,80.0,4480246.0,5163676.0,26.064651
